# Fuzz Archives: Artist-Centered Graph (Hardcore Filter)

*Powered by **Traverse** and Visualized with **Cosmograph***

Same as the artist-centered graph, but filtered to **only** include artists
whose styles include **Hardcore** or **Happy Hardcore**.  Uses the
`require_tags` parameter to pre-filter nodes before edge building.

## 1. Configuration

In [5]:
from pathlib import Path

RECORDS_CSV = Path(r"C:\Users\xtrem\Documents\Datasets\records.csv")
OUT_DIR = Path("_out_artist_cloudrap")
FORCE = True  # set True to rebuild cache

# Tag filter — keep only artists with these styles
REQUIRE_TAGS = {"styles": ["Cloud Rap"]}

# Tuning parameters
MAX_NODES = 0               # 0 = unlimited
MIN_SHARED_TAGS = 1         # min shared tags to create an edge
MAX_EDGES = 0               # 0 = unlimited
MAX_EDGES_PER_NODE = 2      # keep only top-K edges per artist
MAX_TAG_DEGREE = 200       # sample tags shared by more artists than this
SAMPLE_HIGH_DEGREE = True   # True = sample down; False = skip entirely

# When True and multiple tag_types are used, require shared tags in
# EVERY tag type (AND logic) instead of ANY tag type (OR logic).
REQUIRE_ALL_TAG_TYPES = True

## 2. Build or Load Graph

In [6]:
from traverse.graph.artist_graph import build_artist_graph
from traverse.graph.cache import GraphCache

cache = GraphCache(
    cache_dir=OUT_DIR,
    build_fn=lambda: build_artist_graph(
        RECORDS_CSV,
        max_nodes=MAX_NODES,
        min_shared_tags=MIN_SHARED_TAGS,
        max_edges=MAX_EDGES,
        max_edges_per_node=MAX_EDGES_PER_NODE,
        max_tag_degree=MAX_TAG_DEGREE,
        sample_high_degree=SAMPLE_HIGH_DEGREE,
        require_tags=REQUIRE_TAGS,
        require_all_tag_types=REQUIRE_ALL_TAG_TYPES,
    ),
    force=FORCE,
)
graph, records_df = cache.load_or_build()

n_pts = len(graph["points"])
n_lks = len(graph["links"])
total_items = n_pts + n_lks
print(f"Graph: {n_pts:,} nodes, {n_lks:,} edges ({total_items:,} total items)")
print(f"Records: {len(records_df):,} rows")

Building graph (this may take a while)…
Reading records: 74chunk [11:46,  9.55s/chunk]
Pass 1a: 14,609,075 rows scanned, 2,851,681 unique artists with tags (edge tag_types=['styles'])
  require_tags filter: 2,851,681 → 2,561 artists (require={'styles': ['Cloud Rap']})
Finding neighbors: 100%|██████████| 2561/2561 [00:00<00:00, 4377.12node/s]
Pass 2 (nearest-neighbor): 5,114 unique edges, 12 tags sampled, 0 tags skipped (degree > 200)
Pass 3: 2,561 artist nodes, 5,114 edges (min_shared_tags=1, max_edges=0, max_edges_per_node=2)


Graph: 2,561 nodes, 5,114 edges (7,675 total items)
Records: 2,561 rows


Cached graph → _out_artist_cloudrap\graph.json
Cached records → _out_artist_cloudrap\canonical_plays.parquet (2,561 rows)


## 3. Community Detection

In [7]:
from collections import Counter
from traverse.graph.community import add_communities, CommunityAlgorithm

graph = add_communities(graph, CommunityAlgorithm.LOUVAIN, seed=42)

comm_counts = Counter(pt["community"] for pt in graph["points"])
print(f"{len(comm_counts)} communities:")
for comm_id, count in comm_counts.most_common():
    print(f"  Community {comm_id}: {count} nodes")

13 communities:
  Community 0: 394 nodes
  Community 1: 375 nodes
  Community 2: 362 nodes
  Community 3: 297 nodes
  Community 4: 220 nodes
  Community 5: 213 nodes
  Community 6: 187 nodes
  Community 7: 164 nodes
  Community 8: 94 nodes
  Community 9: 87 nodes
  Community 10: 84 nodes
  Community 11: 68 nodes
  Community 12: 16 nodes


## 4. Sanity Check

In [8]:
import random

sample_pt = random.choice(graph["points"])
sample_id = sample_pt["id"]
print(f"Sample node: {sample_pt}")
print()

# Find neighbors
id_to_pt = {pt["id"]: pt for pt in graph["points"]}
neighbors = []
for lk in graph["links"]:
    w = lk.get("weight", 1)
    if lk["source"] == sample_id:
        neighbors.append((lk["target"], w))
    elif lk["target"] == sample_id:
        neighbors.append((lk["source"], w))

neighbors.sort(key=lambda x: x[1], reverse=True)
print(f"{len(neighbors)} neighbors (top 10 by shared tags):")
for nid, w in neighbors[:10]:
    npt = id_to_pt.get(nid, {})
    print(f"  w={w}: {npt.get('label', nid)}")

Sample node: {'id': 'Black Kray; Kane Grocerys; MFK (2)', 'label': 'Black Kray; Kane Grocerys; MFK (2)', 'genres': 'Hip Hop', 'styles': 'Cloud Rap | Trap', 'external_links': [{'platform': 'discogs', 'url': 'https://www.discogs.com/search/?q=Black+Kray%3B+Kane+Grocerys%3B+MFK+%282%29&type=artist', 'label': 'Discogs'}], 'community': 1}

2 neighbors (top 10 by shared tags):
  w=2: Shisa
  w=2: Bopapocalypse.


## 5. Export & Serve

In [9]:
from traverse.graph.adapters_cosmograph import CosmographAdapter
from traverse.cosmograph.server import serve, _default_dist_dir

n_pts = len(graph["points"])
n_lks = len(graph["links"])
print(f"Graph: {n_pts:,} nodes, {n_lks:,} edges")

dataloc = "cosmo_artists_cloudrap.json" 

if n_lks > 500_000:
    print(f"WARNING: {n_lks:,} edges is too many for browser visualization.")
    print("Consider increasing MIN_SHARED_TAGS or lowering MAX_EDGES/MAX_NODES.")
    print("Skipping export. Adjust params and re-run.")
else:
    meta = {"clusterField": "community", "title": "Fuzz Archives — Hardcore"}
    out_path = _default_dist_dir() / dataloc
    CosmographAdapter.write(graph, out_path, meta=meta)
    print()
    print("Starting server — open in browser:")
    print(f"  Local:   http://127.0.0.1:8080/?data=/{dataloc}")
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        local_ip = s.getsockname()[0]
        s.close()
        print(f"  Network: http://{local_ip}:8080/?data=/{dataloc}")
    except Exception:
        print(f"  Network: http://<your-local-ip>:8080/?data=/{dataloc}")
    print()
    print("Press Ctrl+C (or interrupt the kernel) to stop.")
    serve(port=8080, host="0.0.0.0")

Graph: 2,561 nodes, 5,114 edges

Starting server — open in browser:
  Local:   http://127.0.0.1:8080/?data=/cosmo_artists_cloudrap.json
  Network: http://192.168.68.58:8080/?data=/cosmo_artists_cloudrap.json

Press Ctrl+C (or interrupt the kernel) to stop.
Serving C:\Users\xtrem\Documents\Projects\GitHub\traverse_vc\src\traverse\cosmograph\app\dist at http://0.0.0.0:8080
Network access: http://192.168.68.58:8080


Wrote C:\Users\xtrem\Documents\Projects\GitHub\traverse_vc\src\traverse\cosmograph\app\dist\cosmo_artists_cloudrap.json (1.5 MB)
127.0.0.1 - - [28/Feb/2026 13:56:41] "GET /?data=/cosmo_artists_cloudrap.json HTTP/1.1" 200 -
127.0.0.1 - - [28/Feb/2026 13:56:41] "GET /assets/index-Lf-sjV6g.css HTTP/1.1" 304 -
127.0.0.1 - - [28/Feb/2026 13:56:41] "GET /assets/index-DJWoCdHy.js HTTP/1.1" 304 -
127.0.0.1 - - [28/Feb/2026 13:56:41] "GET /api/corrections HTTP/1.1" 200 -
127.0.0.1 - - [28/Feb/2026 13:56:41] code 404, message File not found
127.0.0.1 - - [28/Feb/2026 13:56:41] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [28/Feb/2026 13:56:41] "GET /cosmo_artists_cloudrap.json HTTP/1.1" 200 -
127.0.0.1 - - [28/Feb/2026 13:56:58] "POST /api/album-tracks HTTP/1.1" 200 -
127.0.0.1 - - [28/Feb/2026 13:57:08] "POST /api/album-tracks HTTP/1.1" 200 -
127.0.0.1 - - [28/Feb/2026 13:57:09] "POST /api/album-tracks HTTP/1.1" 200 -
127.0.0.1 - - [28/Feb/2026 13:57:10] "POST /api/album-tracks HTTP/1.1" 200


Shutting down.
